# Horizon total return analysis

**Question this answers:** "If I hold this instrument for 3 months and spreads widen 25bp, what is my total return?"

Horizon total return composes carry, roll-down, and scenario-driven P&L
into a single return under user assumptions. It bridges the scenario engine
and P&L attribution framework: you supply a `ScenarioSpec` (which may include
a time-roll and market shocks), and the analyzer produces a factor-decomposed
total return.

## Setup

We build a 5-year USD corporate bond with a discount curve and a hazard curve,
then run horizon analysis under different scenarios.

In [1]:
import json
from datetime import date

from finstack.scenarios import compute_horizon_return, HorizonResult
from finstack.core.market_data import DiscountCurve, HazardCurve, MarketContext

In [2]:
# 5-year 5% semi-annual corporate bond
bond_json = json.dumps({
    "type": "bond",
    "spec": {
        "id": "CORP-5Y-USD",
        "notional": {"amount": "1000000", "currency": "USD"},
        "issue_date": "2025-01-15",
        "maturity": "2030-01-15",
        "cashflow_spec": {
            "Fixed": {
                "coupon_type": "Cash",
                "rate": 0.05,
                "freq": {"count": 6, "unit": "months"},
                "dc": "Thirty360",
                "bdc": "following",
                "calendar_id": "weekends_only",
                "stub": "None",
                "end_of_month": False,
                "payment_lag_days": 0
            }
        },
        "discount_curve_id": "USD-OIS",
        "credit_curve_id": "CORP-HAZARD",
        "call_put": None,
        "attributes": {"tags": [], "meta": {}},
        "pricing_overrides": {}
    }
})

print("Instrument ready: 5Y 5% USD corporate bond")

Instrument ready: 5Y 5% USD corporate bond


In [3]:
as_of = "2025-01-15"
base_date = date.fromisoformat(as_of)

# Build market context with discount and hazard curves
mc = MarketContext()

disc = DiscountCurve(
    "USD-OIS", base_date,
    [(0.0, 1.0), (0.5, 0.980), (1.0, 0.960), (2.0, 0.920),
     (3.0, 0.880), (5.0, 0.800), (10.0, 0.650)],
    day_count="act_365f",
)

haz = HazardCurve(
    "CORP-HAZARD", base_date,
    [(0.5, 0.010), (1.0, 0.012), (2.0, 0.015),
     (3.0, 0.018), (5.0, 0.022), (10.0, 0.028)],
    day_count="act_365f",
)

mc.insert(disc)
mc.insert(haz)

print("Market ready: USD-OIS discount + CORP-HAZARD credit curves")

Market ready: USD-OIS discount + CORP-HAZARD credit curves


## Scenario 1: Pure carry

Hold the bond for 3 months, assuming markets don't move. All P&L comes from
coupon accrual, pull-to-par, and roll-down.

In [4]:
carry_scenario = json.dumps({
    "id": "carry-3m",
    "name": "3-Month Carry",
    "description": "Hold for 3 months, no market shocks",
    "operations": [
        {
            "kind": "time_roll_forward",
            "period": "3M",
            "apply_shocks": False,
            "roll_mode": "business_days"
        }
    ],
    "priority": 0,
    "resolution_mode": "most_specific_wins"
})

result = compute_horizon_return(bond_json, mc, as_of, carry_scenario)
print(result.explain())

Horizon Total Return: 0.1470%
Annualized: 0.5975%
Horizon: 90 days
Initial Value: USD 976036.09
Terminal Value: USD 977470.84
Total P&L: USD 1434.75
  Carry: USD 11162.80
  Rates: USD -9728.05
  Credit: USD -0.00
  Residual: USD -0.00



In [5]:
print(f"Horizon:          {result.horizon_days} days")
print(f"Total return:     {result.total_return_pct * 100:.4f}%")
print(f"Annualized:       {result.annualized_return * 100:.4f}%")
print(f"Carry component:  {result.factor_contribution('carry') * 100:.4f}%")

Horizon:          90 days
Total return:     0.1470%
Annualized:       0.5975%
Carry component:  1.1437%


## Scenario 2: Rate shock (no holding period)

What if rates rise 50bp right now? No time-roll, so carry is zero
and the result is a pure mark-to-scenario.

In [6]:
shock_scenario = json.dumps({
    "id": "rate-shock-50",
    "name": "Rates +50bp",
    "description": "Instantaneous parallel rate shock",
    "operations": [
        {
            "kind": "curve_parallel_bp",
            "curve_kind": "discount",
            "curve_id": "USD-OIS",
            "discount_curve_id": None,
            "bp": 50.0
        }
    ],
    "priority": 0,
    "resolution_mode": "most_specific_wins"
})

result = compute_horizon_return(bond_json, mc, as_of, shock_scenario)
print(result.explain())
print(f"Horizon days:     {result.horizon_days}  (None = no time-roll)")
print(f"Annualized:       {result.annualized_return}  (None = not meaningful)")

Horizon Total Return: -1.9906%
Initial Value: USD 976036.09
Terminal Value: USD 956606.65
Total P&L: USD -19429.44
  Carry: USD 0.00
  Rates: USD -19429.44
  Credit: USD 0.00
  Residual: USD 0.00

Horizon days:     None  (None = no time-roll)
Annualized:       None  (None = not meaningful)


## Scenario 3: Hold 3 months + spreads widen 25bp

The core use case. Combines a holding period with a market view:
earn carry while credit deteriorates.

In [7]:
combined_scenario = json.dumps({
    "id": "hold-3m-spread-25",
    "name": "3M Hold + Spread Widening",
    "description": "Hold for 3 months while credit spreads widen 25bp",
    "operations": [
        {
            "kind": "time_roll_forward",
            "period": "3M",
            "apply_shocks": True,
            "roll_mode": "business_days"
        },
        {
            "kind": "curve_parallel_bp",
            "curve_kind": "par_cds",
            "curve_id": "CORP-HAZARD",
            "discount_curve_id": "USD-OIS",
            "bp": 25.0
        }
    ],
    "priority": 0,
    "resolution_mode": "most_specific_wins"
})

result = compute_horizon_return(bond_json, mc, as_of, combined_scenario)
print(result.explain())

Horizon Total Return: -0.4723%
Annualized: -1.9015%
Horizon: 90 days
Initial Value: USD 976036.09
Terminal Value: USD 971426.75
Total P&L: USD -4609.33
  Carry: USD 11162.80
  Rates: USD -9666.58
  Credit: USD -6044.08
  Residual: USD -122.93



In [8]:
# Factor contributions as percentage of initial value
factors = ["carry", "rates", "credit", "fx", "vol"]
print("Factor contributions:")
for f in factors:
    pct = result.factor_contribution(f) * 100
    print(f"  {f:12s}  {pct:+.4f}%")

print(f"\n  {'total':12s}  {result.total_return_pct * 100:+.4f}%")
print(f"  annualized:   {result.annualized_return * 100:+.4f}%")

Factor contributions:
  carry         +1.1437%
  rates         -0.9904%
  credit        -0.6192%
  fx            +0.0000%
  vol           +0.0000%

  total         -0.4723%
  annualized:   -1.9015%


## Scenario 4: Multi-factor stress

Hold for 3 months with rates up 100bp and spreads widening 50bp.
A more adverse scenario combining rate and credit moves.

In [9]:
stress_scenario = json.dumps({
    "id": "hold-3m-stress",
    "name": "3M Hold Under Stress",
    "description": "Hold 3 months, rates +100bp, spreads +50bp",
    "operations": [
        {
            "kind": "time_roll_forward",
            "period": "3M",
            "apply_shocks": True,
            "roll_mode": "business_days"
        },
        {
            "kind": "curve_parallel_bp",
            "curve_kind": "discount",
            "curve_id": "USD-OIS",
            "discount_curve_id": None,
            "bp": 100.0
        },
        {
            "kind": "curve_parallel_bp",
            "curve_kind": "par_cds",
            "curve_id": "CORP-HAZARD",
            "discount_curve_id": "USD-OIS",
            "bp": 50.0
        }
    ],
    "priority": 0,
    "resolution_mode": "most_specific_wins"
})

result = compute_horizon_return(bond_json, mc, as_of, stress_scenario)
print(result.explain())

Horizon Total Return: -4.5490%
Annualized: -17.2060%
Horizon: 90 days
Initial Value: USD 976036.09
Terminal Value: USD 931636.21
Total P&L: USD -44399.87
  Carry: USD 11162.80
  Rates: USD -43420.63
  Credit: USD -11416.09
  Residual: USD -1451.89



## Attribution methods

The `method` parameter controls how P&L is decomposed:
- `"parallel"` (default) -- isolate each factor independently
- `"waterfall"` -- sequential application, sum = total by construction
- `"metrics_based"` -- fast linear approximation via DV01/CS01
- `"taylor"` -- sensitivity-based Taylor expansion

In [10]:
for method in ["parallel", "waterfall"]:
    r = compute_horizon_return(bond_json, mc, as_of, combined_scenario, method=method)
    print(f"--- {method} ---")
    print(f"  Total return:  {r.total_return_pct * 100:+.4f}%")
    print(f"  Carry:         {r.attribution.carry:.2f}")
    print(f"  Credit:        {r.attribution.credit_curves_pnl:.2f}")
    print(f"  Residual:      {r.attribution.residual:.2f}")
    print()

--- parallel ---
  Total return:  -0.4723%
  Carry:         11162.81
  Credit:        -6044.09
  Residual:      -122.93

--- waterfall ---
  Total return:  -0.4723%
  Carry:         11162.81
  Credit:        -6044.09
  Residual:      0.00



## Serialization

Results can be serialized to JSON for storage or downstream processing.
The full `PnlAttribution` is available via `.attribution`.

In [11]:
result = compute_horizon_return(bond_json, mc, as_of, combined_scenario)

# Access the full PnlAttribution object
attr = result.attribution
print(f"Attribution method: {attr.method}")
print(f"T0: {attr.t0}  ->  T1: {attr.t1}")
print(f"Residual %: {attr.residual_pct:.4f}%")

# Serialize entire result to JSON
result_json = result.to_json()
print(f"\nJSON output size: {len(result_json)} chars")
print(json.loads(result_json).keys())

Attribution method: Parallel
T0: 2025-01-15  ->  T1: 2025-04-15
Residual %: 2.6671%

JSON output size: 3189 chars
dict_keys(['attribution', 'initial_value', 'terminal_value', 'horizon_days', 'scenario_report'])


## Takeaways

- `compute_horizon_return` composes any `ScenarioSpec` with P&L attribution
- Include a `time_roll_forward` operation for a holding period; omit it for pure mark-to-scenario
- The result wraps a full `PnlAttribution` with convenience accessors for total return, annualized return, and per-factor contributions
- Four attribution methods are available; `"parallel"` is the default